#  ApBot — TensorFlow/Keras Neural Network Training & NLP Pipeline
### SABLE Haute Couture AI Shopping Assistant

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MujtabaZadaii/APBOT_E-commerce/blob/main/apbot/notebooks/ApBot_Training.ipynb)

This notebook trains the **200-Epoch TensorFlow/Keras Deep Learning Intent Classifier** and integrates the **Domain-Aware NLP Typo Correction Layer** for ApBot.

## 1. Google Colab Environment Setup & Auto GitHub Sync
Clones the official GitHub repository automatically if running on Google Colab or remote server.

In [ ]:
import os
import sys

# Auto-detect Google Colab runtime
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB or not os.path.exists('data/intents.json'):
    print(' Setting up GitHub environment for Google Colab...')
    !git clone https://github.com/MujtabaZadaii/APBOT_E-commerce.git repo
    if os.path.exists('repo/apbot'):
        os.chdir('repo/apbot')
    elif os.path.exists('apbot'):
        os.chdir('apbot')
    print(" Repository cloned successfully! Current Directory:", os.getcwd())
else:
    print(" Running in local environment. Current Directory:", os.getcwd())

## 2. Imports & NLTK Resources Download

In [ ]:
import json
import pickle
import random
import re
import difflib
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Download essential NLTK tokenizer and lemmatizer packages
for pkg in ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

lemmatizer = WordNetLemmatizer()
print(" TensorFlow Version:", tf.__version__)
print(" NLTK Resources Ready!")

## 3. Domain-Aware NLP Typo Correction Module
High-speed spell corrector that normalizes user typos (`blak` ➔ `black`, `shrit` ➔ `shirt`, `whre` ➔ `where`) while preserving prices (`£200`), order IDs (`SBL-12345`), emails, and brand terms.

In [ ]:
EXPLICIT_TYPO_MAP = {
    'blak': 'black', 'blck': 'black', 'jaket': 'jacket', 'jackt': 'jacket', 'jact': 'jacket',
    'shrit': 'shirt', 'tsirt': 'shirt', 'dres': 'dress', 'coat': 'coat', 'sweater': 'sweater',
    'trouser': 'trouser', 'trousers': 'trousers', 'whre': 'where', 'wher': 'where', 'wht': 'what',
    'cheep': 'cheap', 'sasta': 'cheap', 'mehnga': 'expensive', 'similer': 'similar', 'trck': 'track',
    'traking': 'tracking', 'ordar': 'order', 'oder': 'order', 'chekout': 'checkout', 'dikhao': 'show',
    'batao': 'tell', 'chahiye': 'want', 'repoart': 'report'
}

DEFAULT_DOMAIN_VOCABULARY = {
    'sable', 'apbot', 'couture', 'luxury', 'black', 'white', 'grey', 'navy', 'ivory', 'charcoal',
    'jacket', 'jackets', 'coat', 'coats', 'shirt', 'shirts', 'tshirt', 'sweater', 'sweaters',
    'trouser', 'trousers', 'tailoring', 'blazer', 'outerwear', 'essentials', 'knitwear', 'clothing',
    'dress', 'outfit', 'men', 'women', 'size', 'sizing', 'fit', 'show', 'find', 'search', 'view',
    'add', 'remove', 'buy', 'cart', 'bag', 'checkout', 'order', 'orders', 'tracking', 'track',
    'parcel', 'price', 'cheap', 'expensive', 'discount', 'wishlist', 'support', 'contact', 'report',
    'new', 'arrivals', 'trending', 'winter', 'office', 'date', 'similar', 'surprise'
}

class DomainSpellCorrector:
    def __init__(self, additional_vocab=None):
        self.vocab = set(DEFAULT_DOMAIN_VOCABULARY)
        if additional_vocab:
            for v in additional_vocab:
                self.vocab.add(v.lower().strip())
        self.explicit_map = EXPLICIT_TYPO_MAP

    def is_protected_token(self, token):
        if re.match(r'^[£$€]?\d+(\.\d+)?%?$', token):
            return True
        if re.match(r'^SBL-[A-Z0-9-]+$', token, re.IGNORECASE):
            return True
        if '@' in token or token.startswith('http') or '.' in token:
            return True
        if len(token) <= 2:
            return True
        return False

    def correct_word(self, word):
        clean_word = word.lower().strip()
        if self.is_protected_token(word):
            return word
        if clean_word in self.explicit_map:
            return self.explicit_map[clean_word]
        if clean_word in self.vocab:
            return word
        if len(clean_word) >= 3:
            matches = difflib.get_close_matches(clean_word, list(self.vocab), n=1, cutoff=0.80)
            if matches:
                return matches[0]
        return word

    def correct_sentence(self, sentence):
        if not sentence or not isinstance(sentence, str):
            return sentence, False
        tokens = re.findall(r'[A-Za-z0-9-£$€@.]+|[^A-Za-z0-9-£$€@.\s]+|\s+', sentence)
        corrected_tokens = []
        was_corrected = False
        for token in tokens:
            if re.match(r'^[A-Za-z0-9-£$€@.]+$', token):
                corrected = self.correct_word(token)
                if corrected.lower() != token.lower():
                    was_corrected = True
                corrected_tokens.append(corrected)
            else:
                corrected_tokens.append(token)
        return "".join(corrected_tokens), was_corrected

spell_corrector = DomainSpellCorrector()
print(" DomainSpellCorrector initialized!")

## 4. Load Intent Dataset (`intents.json`)

In [ ]:
# Path resolution for local vs Colab execution
intents_paths = ['data/intents.json', '../data/intents.json', 'apbot/data/intents.json', 'repo/apbot/data/intents.json']
intents_file = None
for p in intents_paths:
    if os.path.exists(p):
        intents_file = p
        break

if not intents_file:
    raise FileNotFoundError(" Could not locate data/intents.json!")

with open(intents_file, 'r', encoding='utf-8') as f:
    intents_data = json.load(f)

print(f" Loaded {len(intents_data['intents'])} intent tags from {intents_file}")

## 5. Tokenization, Lemmatization & Vocabulary Extraction

In [ ]:
words = []
classes = []
documents = []
ignore_words = ['?', '!', '.', ',']

for intent in intents_data['intents']:
    for pattern in intent['patterns']:
        w_list = nltk.word_tokenize(pattern)
        words.extend(w_list)
        documents.append((w_list, intent['tag']))
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_words]
words = sorted(list(set(words)))
classes = sorted(list(set(classes)))

# Update DomainSpellCorrector with dataset vocabulary
spell_corrector.vocab.update(words)

print(f" Documents (Patterns): {len(documents)}")
print(f" Unique Classes (Intents): {len(classes)}")
print(f" Unique Preprocessed Words: {len(words)}")

## 6. Training Data & Bag-of-Words Vectorization

In [ ]:
training = []
output_empty = [0] * len(classes)

for doc in documents:
    bag = []
    pattern_words = doc[0]
    pattern_words = [lemmatizer.lemmatize(word.lower()) for word in pattern_words]
    
    for w in words:
        bag.append(1 if w in pattern_words else 0)
        
    output_row = list(output_empty)
    output_row[classes.index(doc[1])] = 1
    training.append([bag, output_row])

random.shuffle(training)
training = np.array(training, dtype=object)

train_x = np.array(list(training[:, 0]))
train_y = np.array(list(training[:, 1]))

print(" Feature Vector Shape (train_x):", train_x.shape)
print(" Target One-Hot Shape (train_y):", train_y.shape)

## 7. Build TensorFlow / Keras Neural Network Architecture

In [ ]:
model = Sequential([
    Dense(128, input_shape=(len(train_x[0]),), activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(len(train_y[0]), activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 8. Execute Model Training (200 Epochs)

In [ ]:
history = model.fit(train_x, train_y, epochs=200, batch_size=5, verbose=1)
final_acc = history.history['accuracy'][-1]
print(f"\n🎉 Training Complete! Final Model Accuracy: {final_acc*100:.2f}%")

## 9. Save Model Artifacts (`.keras`, `words.pkl`, `classes.pkl`)

In [ ]:
os.makedirs('model', exist_ok=True)
model.save('model/chatbot_model.keras')
pickle.dump(words, open('model/words.pkl', 'wb'))
pickle.dump(classes, open('model/classes.pkl', 'wb'))

print("✅ Saved model artifacts to model/")
print(" - model/chatbot_model.keras")
print(" - model/words.pkl")
print(" - model/classes.pkl")

## 10. Live Inference & Typo-Tolerant Prediction Test

In [ ]:
def clean_up_sentence(sentence):
    corrected_text, _ = spell_corrector.correct_sentence(sentence)
    sentence_words = nltk.word_tokenize(corrected_text)
    sentence_words = [lemmatizer.lemmatize(word.lower()) for word in sentence_words]
    return sentence_words, corrected_text

def bow(sentence, words_vocab):
    sentence_words, corrected_text = clean_up_sentence(sentence)
    bag = [0] * len(words_vocab)
    for s in sentence_words:
        for i, w in enumerate(words_vocab):
            if w == s:
                bag[i] = 1
    return np.array(bag), corrected_text

def predict_intent(sentence):
    p_bow, corrected_text = bow(sentence, words)
    res = model.predict(np.array([p_bow]), verbose=0)[0]
    results = [[i, r] for i, r in enumerate(res) if r > 0.5]
    results.sort(key=lambda x: x[1], reverse=True)
    
    if not results:
        return {"intent": "unknown", "confidence": float(max(res)), "correctedMessage": corrected_text}
    
    intent_tag = classes[results[0][0]]
    confidence = float(results[0][1])
    
    response_text = ""
    for intent in intents_data['intents']:
        if intent['tag'] == intent_tag:
            response_text = random.choice(intent['responses'])
            break
            
    return {
        "intent": intent_tag,
        "confidence": round(confidence, 4),
        "correctedMessage": corrected_text,
        "response": response_text
    }

# Test scenarios including typos, currency preservation, and new arrivals
test_queries = [
    "Hi there",
    "show me blak jacket under £200",
    "i want a red shrit",
    "whre is my order SBL-12345",
    "New Arrivals",
    "What is your return policy?"
]

print("=" * 70)
print("🧪 APBOT LIVE MODEL INFERENCE EVALUATION")
print("=" * 70)
for query in test_queries:
    out = predict_intent(query)
    print(f"Raw Query   : '{query}'")
    print(f"Corrected   : '{out['correctedMessage']}'")
    print(f"Predicted   : Intent: {out['intent']} | Confidence: {out['confidence']*100:.2f}%")
    print(f"Bot Reply   : {out['response']}")
    print("-" * 70)